In [1]:
# Goal => get rid of all the superficial dependencies and make the code base bare bones

In [1]:
!python --version

Python 3.13.9


The system cannot find the path specified.


In [2]:
# why do these cunts love overcomplicating shit??

from hydra import compose, initialize
from hydra.utils import instantiate
from omegaconf import OmegaConf

import json
import os

In [3]:
os.listdir(r"./configs/sam2/")

['sam2_hiera_b+.json',
 'sam2_hiera_b+.yaml',
 'sam2_hiera_l.json',
 'sam2_hiera_l.yaml',
 'sam2_hiera_s.json',
 'sam2_hiera_s.yaml',
 'sam2_hiera_t.json',
 'sam2_hiera_t.yaml']

In [4]:
os.listdir(r"./configs/sam2.1/")

['sam2.1_hiera_b+.json',
 'sam2.1_hiera_b+.yaml',
 'sam2.1_hiera_l.json',
 'sam2.1_hiera_l.yaml',
 'sam2.1_hiera_s.json',
 'sam2.1_hiera_s.yaml',
 'sam2.1_hiera_t.json',
 'sam2.1_hiera_t.yaml']

In [9]:
hydra_overrides_extra = [
            # dynamically fall back to multi-mask if the single mask is not stable
            "++model.sam_mask_decoder_extra_args.dynamic_multimask_via_stability=true",
            "++model.sam_mask_decoder_extra_args.dynamic_multimask_stability_delta=0.05",
            "++model.sam_mask_decoder_extra_args.dynamic_multimask_stability_thresh=0.98",
        ]

In [20]:
# convert all the .yaml configs to .json
for conf in os.listdir(r"./configs/sam2.1/"):
    if conf.endswith(r".yaml"):
        with initialize(config_path=r"./configs/sam2.1/", job_name="_", version_base=None):
            config = compose(config_name=f"{conf}", overrides=hydra_overrides_extra)
            OmegaConf.resolve(config)
            with open(f"configs/sam2.1/{conf.replace('yaml', 'json')}", mode="w") as fp:
                json.dump(obj=OmegaConf.to_object(config), fp=fp, indent=3)

In [21]:
# do the same for sam2 as well
for conf in os.listdir(r"./configs/sam2/"):
    if conf.endswith(r".yaml"):
        with initialize(config_path=r"./configs/sam2/", job_name="_", version_base=None):
            config = compose(config_name=f"{conf}", overrides=hydra_overrides_extra)
            OmegaConf.resolve(config)
            with open(f"configs/sam2/{conf.replace('yaml', 'json')}", mode="w") as fp:
                json.dump(obj=OmegaConf.to_object(config), fp=fp, indent=3)

In [23]:
config["model"]["sam_mask_decoder_extra_args"]

{'dynamic_multimask_via_stability': True, 'dynamic_multimask_stability_delta': 0.05, 'dynamic_multimask_stability_thresh': 0.98}

In [28]:
config.model

{'_target_': 'sam2.modeling.sam2_base.SAM2Base', 'image_encoder': {'_target_': 'sam2.modeling.backbones.image_encoder.ImageEncoder', 'scalp': 1, 'trunk': {'_target_': 'sam2.modeling.backbones.hieradet.Hiera', 'embed_dim': 144, 'num_heads': 2, 'stages': [2, 6, 36, 4], 'global_att_blocks': [23, 33, 43], 'window_pos_embed_bkg_spatial_size': [7, 7], 'window_spec': [8, 4, 16, 8]}, 'neck': {'_target_': 'sam2.modeling.backbones.image_encoder.FpnNeck', 'position_encoding': {'_target_': 'sam2.modeling.position_encoding.PositionEmbeddingSine', 'num_pos_feats': 256, 'normalize': True, 'scale': None, 'temperature': 10000}, 'd_model': 256, 'backbone_channel_list': [1152, 576, 288, 144], 'fpn_top_down_levels': [2, 3], 'fpn_interp_model': 'nearest'}}, 'memory_attention': {'_target_': 'sam2.modeling.memory_attention.MemoryAttention', 'd_model': 256, 'pos_enc_at_input': True, 'layer': {'_target_': 'sam2.modeling.memory_attention.MemoryAttentionLayer', 'activation': 'relu', 'dim_feedforward': 2048, 'dro

In [22]:
# we should now be able to use the model using simple json configs without all that meta crap!

In [29]:
# do a dummy instantiation without hydra and OmegaConf

from modeling.sam2_base import SAM2Base

In [5]:
with open(r"./configs/sam2.1/sam2.1_hiera_l.json", mode="r") as fp:
    sam_21_hiera_l_conf = json.load(fp=fp)

In [15]:
sam_21_hiera_l_conf["model"]#["memory_encoder"]

{'_target_': 'sam2.modeling.sam2_base.SAM2Base',
 'image_encoder': {'_target_': 'sam2.modeling.backbones.image_encoder.ImageEncoder',
  'scalp': 1,
  'trunk': {'_target_': 'sam2.modeling.backbones.hieradet.Hiera',
   'embed_dim': 144,
   'num_heads': 2,
   'stages': [2, 6, 36, 4],
   'global_att_blocks': [23, 33, 43],
   'window_pos_embed_bkg_spatial_size': [7, 7],
   'window_spec': [8, 4, 16, 8]},
  'neck': {'_target_': 'sam2.modeling.backbones.image_encoder.FpnNeck',
   'position_encoding': {'_target_': 'sam2.modeling.position_encoding.PositionEmbeddingSine',
    'num_pos_feats': 256,
    'normalize': True,
    'scale': None,
    'temperature': 10000},
   'd_model': 256,
   'backbone_channel_list': [1152, 576, 288, 144],
   'fpn_top_down_levels': [2, 3],
   'fpn_interp_model': 'nearest'}},
 'memory_attention': {'_target_': 'sam2.modeling.memory_attention.MemoryAttention',
  'd_model': 256,
  'pos_enc_at_input': True,
  'layer': {'_target_': 'sam2.modeling.memory_attention.MemoryAtten

In [10]:
sam2.modeling.backbones.image_encoder.ImageEncoder(
 scalp = 1,
 trunk = sam2.modeling.backbones.hieradet.Hiera(embed_dim = 144, num_heads = 2, stages = [2, 6, 36, 4], global_att_blocks = [23, 33, 43], window_pos_embed_bkg_spatial_size = [7, 7], window_spec = [8, 4, 16, 8]),
 neck = sam2.modeling.backbones.image_encoder.FpnNeck(
  position_encoding = sam2.modeling.position_encoding.PositionEmbeddingSine(num_pos_feats = 256, normalize = True, scale = None, temperature = 10000),
  d_model = 256,
  backbone_channel_list = [1152, 576, 288, 144],
  fpn_top_down_levels = [2, 3],
  fpn_interp_model = 'nearest')
)

NameError: name 'sam2' is not defined

In [38]:
# _target_ is the class name, and the rest of the key value pairs within that curly braces are arguments to instantiate an object of that class
# e.g. {'_target_': 'sam2.modeling.memory_encoder.CXBlock', 'dim': 256, 'kernel_size': 7, 'padding': 3, 'layer_scale_init_value': 1e-06, 'use_dwconv': True} means

from modeling.memory_encoder import CXBlock
CXBlock(**{'dim': 256, 'kernel_size': 7, 'padding': 3, 'layer_scale_init_value': 1e-06, 'use_dwconv': True}) # that's how it's done :)

CXBlock(
  (dwconv): Conv2d(256, 256, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=256)
  (norm): LayerNorm2d()
  (pwconv1): Linear(in_features=256, out_features=1024, bias=True)
  (act): GELU(approximate='none')
  (pwconv2): Linear(in_features=1024, out_features=256, bias=True)
  (drop_path): Identity()
)

In [ ]:
SAM2Base()